# AeroGuard-X — Explainable Advisories with SHAP

A safety advisory a human must trust cannot be a black box. This notebook shows
how each impairment warning is attributed to the temporal features that drove
it, using SHAP.

> Requires: `pip install -e ".[explain]"`


In [ ]:
import numpy as np
from aeroguard.ml.temporal import compare_temporal_vs_frame
from aeroguard.ml.sequence_dataset import generate_sequence_dataset
from aeroguard.ml.explain import AdvisoryExplainer

# Train a temporal model and wrap it with the SHAP explainer
model, report = compare_temporal_vs_frame(n_windows=4000, seed=1)
explainer = AdvisoryExplainer(model)
print(f"Model ready. Temporal ROC-AUC = {report.temporal_roc_auc:.3f}")

## Explain a high-risk window

We grab an impaired window, score it, and ask SHAP which temporal features
pushed the probability up.

In [ ]:
X, y = generate_sequence_dataset(n_windows=300, seed=99)
danger_idx = np.where(y == 1)[0][0]
window = X[danger_idx]

prob = model.predict_proba_window(window)
print(f"Impairment probability: {prob:.1%}\n")

attrs = explainer.explain_window(window, top_k=6)
for a in attrs:
    print(f"  {a.feature:28s}  value={a.value:8.3f}  {a.direction()}")

print("\nOne-line summary:")
print(" ", explainer.describe(window, top_k=3))

## Why this matters

Instead of an opaque "82% risk", the system can say *why*: e.g. sustained high-g
exposure with falling perfusion. For any human-facing safety tool, that
explainability is often as important as the accuracy itself — an operator needs
to know whether to trust and act on the warning.

This closes the loop: **honest data → temporal model that beats the baseline →
explainable output.**